# FoundML 3 — Week 2 Coding Assignment (Student Version)
## Ensemble Learning (Real Dataset): Voting and Stacking on UCI Human Activity Recognition (HAR)

In this assignment you will demonstrate the value of **combining diverse learners** using:

1. **Single models** (Logistic Regression, linear SVM, k-NN)
2. **Soft Voting** (average predicted probabilities)
3. **Stacking** (a meta-learner learns how to combine base learners)

Dataset: **UCI Human Activity Recognition (HAR)** (smartphone accelerometer/gyroscope features).  
Task: **6-class classification**.

Main metric: **Macro F1** (treats all classes equally).

Difficulty level: **moderate**.

All numerical answers must be rounded to **2 decimals**.


## Rules
- Use the provided random seeds and hyperparameters.
- Do not change the grading variable names (Q1–Q4).
- Keep the pipelines as specified (especially StandardScaler).


## 0. Setup

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

np.random.seed(0)


## 1. Download and load UCI HAR (do not modify)

We use the official UCI HAR dataset. The code below downloads and extracts it locally if needed.


In [ ]:
DATA_DIR = Path("har_data")
OUTER_ZIP_PATH = DATA_DIR / "HAR_outer.zip"

# UCI's download may deliver either:
# - a single zip containing the folder "UCI HAR Dataset/..." directly, OR
# - an outer zip that contains another zip (often named "UCI HAR Dataset.zip").
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1) Download outer zip if needed
if not OUTER_ZIP_PATH.exists():
    print("Downloading HAR dataset archive...")
    urllib.request.urlretrieve(UCI_ZIP_URL, OUTER_ZIP_PATH)
    print("Download complete:", OUTER_ZIP_PATH)

# 2) Extract outer zip (idempotent; safe to rerun)
print("Extracting outer archive (if needed)...")
with zipfile.ZipFile(OUTER_ZIP_PATH, "r") as zf:
    zf.extractall(DATA_DIR)

# 3) If extraction produced nested zip(s), extract them too until we can locate train/X_train.txt
def extract_all_zips_once(base_dir: Path) -> int:
    """Extract all .zip files found under base_dir (excluding the outer zip).
    Returns the number of zip files extracted in this pass."""
    extracted = 0
    for zpath in base_dir.rglob("*.zip"):
        if zpath.resolve() == OUTER_ZIP_PATH.resolve():
            continue
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                zf.extractall(zpath.parent)
            extracted += 1
        except zipfile.BadZipFile:
            pass
    return extracted

def find_har_root(base_dir: Path) -> Path:
    hits = list(base_dir.rglob("train/X_train.txt"))
    if not hits:
        raise FileNotFoundError("Could not locate train/X_train.txt under har_data after extraction.")
    return hits[0].parent.parent  # directory that contains train/ and test/

for pass_id in range(3):
    try:
        EXTRACT_DIR = find_har_root(DATA_DIR)
        break
    except FileNotFoundError:
        n_extracted = extract_all_zips_once(DATA_DIR)
        if n_extracted == 0:
            raise
        print(f"Extracted {n_extracted} nested zip file(s) (pass {pass_id+1}).")

print("Using HAR dataset root:", EXTRACT_DIR)

def load_txt(path):
    return np.loadtxt(path)

X_train = load_txt(EXTRACT_DIR / "train" / "X_train.txt")
y_train = load_txt(EXTRACT_DIR / "train" / "y_train.txt").astype(int).ravel()

X_test  = load_txt(EXTRACT_DIR / "test" / "X_test.txt")
y_test  = load_txt(EXTRACT_DIR / "test" / "y_test.txt").astype(int).ravel()

# Labels are 1..6 in the dataset; convert to 0..5 for scikit-learn convenience
y_train -= 1
y_test  -= 1

X_train.shape, X_test.shape, np.unique(y_train), np.unique(y_test)


## 2. Class names and a quick class balance check

This helps interpret confusion matrices later.


In [ ]:
class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

counts = np.bincount(y_train, minlength=6)
for i, c in enumerate(counts):
    print(f"{i}: {class_names[i]:<18}  n={c}")


## 3. Evaluation helper (do not modify)

We compute:
- 5-fold stratified **CV macro F1** on the training set
- **Test macro F1** and **Test accuracy** on the held-out test set

Note: All models must implement `predict_proba` for soft voting and stacking with `stack_method="predict_proba"`.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def eval_model(model, X_train, y_train, X_test, y_test):
    # CV macro-F1 on the training set
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro").mean()

    # Fit and evaluate on held-out test set
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    test_f1 = f1_score(y_test, pred, average="macro")
    test_acc = accuracy_score(y_test, pred)

    return float(cv_f1), float(test_f1), float(test_acc)


## 4. Single models (baselines)

Build and evaluate the following three models.

### 4.1 Logistic Regression (multinomial)
Use a pipeline:
- `StandardScaler()`
- `LogisticRegression(max_iter=1000, solver="lbfgs", random_state=0)`

### 4.2 linear SVM (probability output enabled)
Use a pipeline:
- `StandardScaler()`
- `SVC(kernel="linear", C=0.5, probability=True, random_state=0)`

### 4.3 k-NN
Use a pipeline:
- `StandardScaler()`
- `KNeighborsClassifier(n_neighbors=25)`

**Task:** Create the three models and compute their metrics using `eval_model`.
Store results in:
- `lr_cv_f1`, `lr_test_f1`, `lr_test_acc`
- `svm_cv_f1`, `svm_test_f1`, `svm_test_acc`
- `knn_cv_f1`, `knn_test_f1`, `knn_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define lr, svm, knn (pipelines as specified)
# - evaluate each using eval_model
# - define all 9 variables listed above
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
required = [
    "lr_test_f1","svm_test_f1","knn_test_f1",
    "lr_cv_f1","svm_cv_f1","knn_cv_f1"
]
for r in required:
    assert r in globals(), f"{r} not defined"
for v in [lr_test_f1, svm_test_f1, knn_test_f1]:
    assert 0.0 <= v <= 1.0, "F1 should be in [0, 1]"
print("Sanity check passed: single-model scores look valid.")


### **Question 1**
Which **single model** achieved the highest **test macro F1**?

Set `Q1_best_single_model` to exactly one of:
- `"lr"`
- `"svm"`
- `"knn"`


In [ ]:
# ANSWER Q1
# Write your own code:
# - compare lr_test_f1, svm_test_f1, knn_test_f1
# - set Q1_best_single_model
#
# YOUR CODE HERE


## 5. Soft Voting Ensemble

Create a **soft voting** ensemble of the three base models (LR, SVM, kNN).

Use:
- `VotingClassifier(..., voting="soft")`

**Task:** Define the voting model and compute:
- `vote_cv_f1`, `vote_test_f1`, `vote_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define voting (soft voting) using the three models (lr, svm, knn)
# - compute vote_cv_f1, vote_test_f1, vote_test_acc using eval_model
#
# YOUR CODE HERE


### **Question 2**
What is the **test macro F1** of the **soft voting** ensemble? (rounded to 2 decimals)

Define:
- `Q2_vote_test_f1`


In [ ]:
# ANSWER Q2
# YOUR CODE HERE


## 6. Stacking Ensemble

Create a stacking model using the same base learners, and a logistic regression meta-learner.

Use:
- `final_estimator = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=0)`
- `passthrough = False`
- `stack_method = "predict_proba"`
- `cv = 3` (to keep runtime reasonable)

**Task:** Define the stacking model and compute:
- `stack_cv_f1`, `stack_test_f1`, `stack_test_acc`


In [ ]:
# Write here your own code.
# Requirements:
# - define stacking using lr, svm, knn as base estimators
# - final_estimator as specified above
# - stack_method="predict_proba"
# - passthrough=False
# - cv=3
# - compute stack_cv_f1, stack_test_f1, stack_test_acc using eval_model
#
# YOUR CODE HERE


### **Question 3**
What is the **test macro F1** of the **stacking** ensemble? (rounded to 2 decimals)

Define:
- `Q3_stack_test_f1`


In [ ]:
# ANSWER Q3
# YOUR CODE HERE


### **Question 4**
Which of these achieved the highest **test macro F1**?

Choose one of:
- `"best_single"` (the best of lr/svm/knn)
- `"voting"`
- `"stacking"`

Define:
- `Q4_best_overall`


In [ ]:
# ANSWER Q4
# Write your own code.
# Requirements:
# - compare best single test F1 vs vote_test_f1 vs stack_test_f1
# - define Q4_best_overall
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'Q4_best_overall' in globals(), "Q4_best_overall not defined"
assert Q4_best_overall in ["best_single","voting","stacking"], "Invalid choice for Q4_best_overall"
print("Sanity check passed: Q4_best_overall =", Q4_best_overall)


## 7. Visualization (not graded)

1. Confusion matrices on the **test set** for:
   - best single model
   - soft voting
   - stacking
2. Bar chart comparing **CV macro F1** across:
   - LR, SVM, kNN, Voting, Stacking


In [ ]:
import matplotlib.pyplot as plt

# Write here your own code.
# Requirements:
# - Fit the models you need
# - Plot confusion matrices using ConfusionMatrixDisplay.from_estimator
# - Bar chart for CV F1 values
#
# YOUR CODE HERE


## Submission checklist

Your notebook must define:
- `Q1_best_single_model`
- `Q2_vote_test_f1`
- `Q3_stack_test_f1`
- `Q4_best_overall`
